In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from openpyxl import load_workbook
import datetime as dt
from openpyxl import Workbook
import seaborn as sns
import mysql.connector
from sqlalchemy import create_engine
import itertools
from scipy.signal import savgol_filter

In [69]:
from functions import *

In [3]:
#############################################################################################################
#directorios historico de siembras
dir_ = "//172.10.18.2/usuarios/Hilder/Data/tabla_curvas_clavel.xlsx"
#hoja
sh_ = "Hoja1"
#nombre de columnas
col_ = ['finca','bloque','cosecha','nvariedad','cflor','edad','lunes','matas','corte']
con_ = {'finca':str,'cosecha':str}

dir_peaks = "//svrinverpalmas/usuarios/Hilder/Data/tabla_picos.xlsx"
#hoja
sh_peaks = "Hoja1"
#nombre de columnas
col_peaks = ['edad','pico']
con_peaks = {}

# Datos del excel curvas
ruta_archivo = r'\\svrinverpalmas\usuarios\Hilder\Data\tika-curvas2025.csv'

# Leer el archivo con la codificación adecuada
df_cp = pd.read_csv(ruta_archivo, encoding='latin1')

# Seleccionar y renombrar las columnas necesarias
df_cp = df_cp[['Variedad', 'Numero de Semana', 'ProducciÃ³n']]
df_cp.rename(
    columns={'Variedad': 'nvariedad', 'Numero de Semana': 'edad', 'ProducciÃ³n': 'fm'},
    inplace=True
)

# Reemplazar guiones bajos por espacios en la columna 'variedad'
df_cp['nvariedad'] = df_cp['nvariedad'].str.replace('_', ' ')

In [4]:
df_= create_DF(dir_,sh_,col_,con_)

In [5]:
df_ = df_.rename({'lunes':'fecha'}, axis=1)

In [6]:
#df_ = df_.query("nvariedad=='FRONTERA'")

In [7]:
#df_cp = create_DF(dir_cp,sh_cp,col_cp,con_cp)
df_cp.rename({'fm':'fm_proy'},axis=1,inplace=True)

In [8]:
df_['cosecha'] = df_['cosecha'].fillna('')  # Reemplaza los valores NaN con una cadena vacía
df_ = df_[df_['cosecha'].str.contains('25|23|24')]

In [9]:
df_peaks= create_DF(dir_peaks,sh_peaks,col_peaks,con_peaks)

In [10]:
df = pd.merge(df_,df_peaks, on='edad', how='left')

df = df.query("pico > 0 and pico < 3")
# Calcular la columna 'f_m'
df['f_m'] = df['corte'] / df['matas']
    
# Eliminar las columnas 'matas' y 'corte'
df.drop(['matas', 'corte'], inplace=True, axis=1)

In [11]:
# Filtrar el DataFrame solo tercdfos
#df = df.loc[~((df['cosecha'].astype(str).str.contains('18|19|20|21')) & (df['pico'] < 3))]

In [12]:
(
    df.groupby(['finca', 'bloque', 'nvariedad', 'cosecha','pico'])['f_m'].sum().reset_index()
    .query("f_m >= 0 and pico==2")['f_m']
    .quantile(0.15)
)

0.4756034925526451

In [13]:
(
    df.groupby(['finca', 'bloque', 'nvariedad', 'cosecha','pico'])['f_m'].sum().reset_index()
    .query("f_m >= 0 and pico==2")['f_m']
    .quantile(0.85)
)

6.56998556998557

In [79]:
(
    df.groupby(['finca', 'bloque', 'nvariedad', 'cosecha','pico'])['f_m'].sum().reset_index()
    .query("pico==2")
)[:2]


,finca,bloque,nvariedad,cosecha,pico,f_m
1,001,16,ANASTASIA,AGO24,2.0,5.360160
2,001,16,ANASTASIA,EAS23,2.0,0.243452


## filtrar las que aplican y las que no

In [15]:
# Lista para almacenar los resultados finales
final_dfs = []

# Valores de pico (supongamos que son 1, 2 y 3)
picos = [1,2]

for pico in picos:
    # Filtrar el dataframe original por el valor de pico actual
    df_filtered = df[df['pico'] == pico]
    
    # Seleccionar las columnas relevantes
    df_filtered = df_filtered[['finca','bloque','nvariedad','cosecha','f_m','edad']]
    
    # Agrupar y sumar la columna 'f_m' por finca, bloque, nvariedad, cosecha
    grouped_df = df_filtered.groupby(['finca', 'bloque', 'nvariedad', 'cosecha'])['f_m'].sum().reset_index()
    
    grouped_df = grouped_df.query("f_m >= 3.5")
    
    # Crear una lista para almacenar los dataframes filtrados por cada variedad
    filtered_dfs = []

    for variedad in grouped_df['nvariedad'].unique():
        # Filtrar por la variedad actual
        variedad_df = grouped_df[grouped_df['nvariedad'] == variedad]

        # Calcular el rango intercuartílico por variedad
        q1 = variedad_df['f_m'].quantile(0.15)
        q3 = variedad_df['f_m'].quantile(0.85)

        # Filtrar los grupos dentro del rango intercuartílico para la variedad actual
        filtered_variedad_df = variedad_df[(variedad_df['f_m'] >= q1) & (variedad_df['f_m'] <= q3)]

        # Agregar el dataframe filtrado a la lista
        filtered_dfs.append(filtered_variedad_df)

    # Concatenar los dataframes filtrados en uno solo
    filtered_df = pd.concat(filtered_dfs)

    # Filtrar el dataframe original en base a los grupos filtrados
    new_df = pd.merge(df_filtered, filtered_df, on=['finca', 'bloque', 'nvariedad', 'cosecha'], how='left')
    new_df = new_df.rename({'f_m_x':'f_m'}, axis=1)
    new_df['f_m_y'] = new_df['f_m_y'].apply(lambda x: 0 if pd.isna(x) else 1)
    new_df = new_df.rename({'f_m_y':'filtro'}, axis=1)
    
    # Agregar el resultado a la lista final
    final_dfs.append(new_df)

# Concatenar los resultados de los tres picos en un solo dataframe
final_result = pd.concat(final_dfs)
final_result = pd.merge(final_result,df_peaks,on='edad',how='left')

In [16]:
new_dfR = final_result.copy()
new_dfR = new_dfR.query("filtro == 1")

In [17]:
# Crear el nuevo DataFrame vacío donde se guardarán los resultados
nuevo_df = pd.DataFrame()

df_filtrado = new_dfR.groupby(['finca', 'bloque', 'nvariedad', 'cosecha', 'edad', 'pico', 'filtro'])['f_m'].mean().reset_index()

# Agrupar el DataFrame original por finca, cosecha, bloque y nvariedad
grupos = df_filtrado.groupby(['finca', 'cosecha', 'bloque', 'nvariedad', 'pico'])

# Iterar sobre cada grupo y aplicar la función info_vitrinas
for grupo, data in grupos:
    finca, cosecha, bloque, nvariedad, pico = grupo
    try:
        resultado = info_picos(data, finca, cosecha, bloque, nvariedad, pico)  # Llama a la función info_vitrinas
    except:
        continue  # Si ocurre una excepción, salta al siguiente grupo
    
    # Agregar el resultado al nuevo DataFrame
    if resultado is not None:
        nuevo_df = pd.concat([nuevo_df, resultado], ignore_index=True)
final_df = nuevo_df.copy()

In [18]:
# Filter out rows with sem_pico equal to 0
moda_df = final_df[final_df['sem_pico'] != 0]

# Calculate the mode of sem_pico grouped by nvariedad
moda_sem_pico = moda_df.groupby(['nvariedad','pico'])['sem_pico'].agg(lambda x: x.mode().iloc[0] if not x.empty else 0).reset_index()

In [19]:
df_rev_picos = final_df.groupby(
    ['finca','bloque','nvariedad','cosecha','edad','pico','sem_pico','filtro']
)['f_m'].mean().reset_index().drop('f_m',axis=1).query("sem_pico > 0")

In [20]:
df_rev_picos.to_excel(
    "C:/Users/Estadistica/Downloads/edad que alcanza pico cada variedad.xlsx"
)

In [21]:
df_rev_picos.head()

,finca,bloque,nvariedad,cosecha,edad,pico,sem_pico,filtro
14,001,16,ANASTASIA,AGO24,60,2,60.0,1
24,001,16,ANTIGUA,FEB25,28,1,28.0,1
43,001,16,BERNARD,NOV23,54,2,54.0,1
56,001,16,BERNARD,SVA24,25,1,25.0,1
77,001,16,BERNARD,SVA24,53,2,53.0,1


In [80]:
final_result.query("nvariedad=='CAROLINE' ").head()

,finca,bloque,nvariedad,cosecha,f_m,edad,filtro,pico
4177,001,23,CAROLINE,ABR25,0.030303,20,1,1
4178,001,23,CAROLINE,ABR25,0.175275,21,1,1
4179,001,23,CAROLINE,ABR25,0.696281,22,1,1
4180,001,23,CAROLINE,ABR25,1.028237,23,1,1
4181,001,23,CAROLINE,ABR25,1.534780,24,1,1


In [23]:
df_agrupado = pd.merge(df_rev_picos,moda_sem_pico,on=['nvariedad','pico'],how='left')

In [24]:
moda_sem_pico.to_excel("//svrinverpalmas/usuarios/Hilder/Data/moda sem pico.xlsx",index=False)

In [25]:
new_df_1 = df_agrupado.copy()

# Aplica la función a cada fila del DataFrame
new_df_1['filtro'] = new_df_1.apply(calcular_filtro, axis=1)

# Convierte la columna "bloque" a tipo int64
new_df_1['bloque'] = new_df_1['bloque'].astype('int64')

In [26]:
new_df_1.query("nvariedad=='ROMEO' and pico==1")[:1]

,finca,bloque,nvariedad,cosecha,edad,pico,sem_pico_x,filtro,sem_pico_y
428,001,26,ROMEO,MAY25,27,1,27.0,2,27.0


In [27]:
new_df_1.groupby(
    ['filtro']
).size()

filtro
1      32
2    2343
dtype: int64

Agrupar por variedad

#Unir datos finales para mostrar todo

In [28]:
df_0 = df.copy()
df_0 = df_0[['finca','bloque','nvariedad','cosecha','pico','edad','f_m']]

In [29]:
df_1 = new_df_1.copy()
df_1 = df_1[['finca','bloque','nvariedad','cosecha','pico','sem_pico_x','sem_pico_y','filtro']]

In [30]:
df_0.head()

,finca,bloque,nvariedad,cosecha,pico,edad,f_m
0,001,23,AMICO LAVANDER,FEB24,1.0,22,0.010326
1,001,23,AMICO LAVANDER,FEB24,1.0,23,0.107609
2,001,23,AMICO LAVANDER,FEB24,1.0,24,0.215217
3,001,23,AMICO LAVANDER,FEB24,1.0,25,0.448913
4,001,23,AMICO LAVANDER,FEB24,1.0,26,0.533152


In [31]:
df_2 = pd.merge(
    df_0,
    df_1,
    on= ['finca','bloque','nvariedad','cosecha','pico'],
    how = 'left'
)

In [32]:
def alexander(row):
    return ((row['sem_pico_x']-5)*2)

In [33]:
df_prueba_A = df_2[(df_2.sem_pico_x.notna()) & (df_2.pico==1)].reset_index()
df_prueba_A['incia_2do'] = df_prueba_A.apply(
    alexander,
    axis=1
)

In [34]:
df_2.head()

,finca,bloque,nvariedad,cosecha,pico,edad,f_m,sem_pico_x,sem_pico_y,filtro
0,001,23,AMICO LAVANDER,FEB24,1.0,22,0.010326,29.0,28.0,2.0
1,001,23,AMICO LAVANDER,FEB24,1.0,23,0.107609,29.0,28.0,2.0
2,001,23,AMICO LAVANDER,FEB24,1.0,24,0.215217,29.0,28.0,2.0
3,001,23,AMICO LAVANDER,FEB24,1.0,25,0.448913,29.0,28.0,2.0
4,001,23,AMICO LAVANDER,FEB24,1.0,26,0.533152,29.0,28.0,2.0


In [35]:
df_2.to_excel("//svrinverpalmas/usuarios/Hilder/Data/curvas ajustadas.xlsx")

In [36]:
"""
df_2['sem_pico_x'].fillna(0, inplace=True)  # Replace NaN with 0
df_2['sem_pico_y'].fillna(0, inplace=True)  # Replace NaN with 0

df_2['sem_pico_x'] = df_2['sem_pico_x'].astype('int')
df_2['sem_pico_y'] = df_2['sem_pico_y'].astype('int')

df_2.head()"""
df_2['filtro'].fillna(0, inplace=True)  # Replace NaN with 0
df_2['filtro'] = df_2['filtro'].astype('int')

In [37]:
df_3 = df_2.query("filtro!=1").pivot_table(
    index = ['finca','nvariedad','edad','pico'],
    columns = 'filtro',
    values = 'f_m',
    aggfunc=np.mean
).reset_index()

In [38]:
df_4 = df_2.query("filtro!=1").pivot_table(
    index = ['nvariedad','edad','pico'],
    columns = 'filtro',
    values = 'f_m',
    aggfunc=np.mean
).reset_index()
df_4['finca'] = '000'

In [39]:
df_5 = pd.concat(
    [df_3,df_4],
    axis=0
)
df_5 = df_5.rename(columns={0: 'fm_real', 2: 'fm_aj'})

In [40]:
df_6 = pd.merge(
    df_5,
    df_cp,
    on = ['nvariedad','edad'],
    how='left'
)[['finca','nvariedad','edad','pico','fm_real','fm_proy','fm_aj']]

df_6 = df_6.fillna(0)

In [86]:
df_2.query("nvariedad=='CAROLINE'").groupby(['finca','bloque','nvariedad','cosecha','pico','filtro'])['f_m'].sum().reset_index().to_excel("C:/Users/Estadistica/Downloads/caroline.xlsx")

In [42]:
df_g = df_2.query("pico==1 and filtro==2 and nvariedad=='DON PEDRO'").groupby(['edad','filtro','pico'])['f_m'].mean().reset_index()

In [43]:
df_6 = df_6.query("edad>=18 and edad <= 70")

In [44]:
df_6.drop('pico',axis=1,inplace=True)

In [45]:
agrupaciones = ['finca', 'nvariedad']
edades = np.arange(18, 71)

# Generar todas las combinaciones posibles de agrupaciones y edades
combinaciones = list(itertools.product(*(df_6[column].unique() for column in agrupaciones), edades))
df_combinaciones = pd.DataFrame(combinaciones, columns=agrupaciones + ['edad'])

# Agregar la columna "flor_mata" con valor cero en el DataFrame de combinaciones
df_combinaciones['edad'] = df_combinaciones.edad.astype('int64')

In [46]:
df_7 = pd.merge(
    df_combinaciones,
    df_6,
    on = ['finca','nvariedad','edad'],
    how='left'
)
df_7 = df_7.fillna(0)

In [47]:
df_7['pico'] = df_7.edad.apply(coloca_pico)

In [77]:
df_7.query("nvariedad=='CAROLINE'").shape

(159, 7)

In [82]:
df_processed = procesar_curva(df_7.query("nvariedad=='CAROLINE'"), '001', 'CAROLINE')

IndexError: index 50 is out of bounds for axis 0 with size 50

In [83]:
# Or process all combinations at once
df_all_processed = procesar_todas_curvas(df_7)

Error processing finca 001, variety ARNO: 'NoneType' object is not subscriptable
Error processing finca 001, variety CAROLINE: index 50 is out of bounds for axis 0 with size 50
Error processing finca 001, variety CAVA: 'NoneType' object is not subscriptable
Error processing finca 001, variety EMMA: 'NoneType' object is not subscriptable
Error processing finca 001, variety LAILA: 'NoneType' object is not subscriptable
Error processing finca 001, variety MARTINA: 'NoneType' object is not subscriptable
Error processing finca 001, variety MONSENOR: 'NoneType' object is not subscriptable
Error processing finca 001, variety MOZZART: index 50 is out of bounds for axis 0 with size 50
Error processing finca 001, variety NIMBUS: 'NoneType' object is not subscriptable
Error processing finca 001, variety NUMBER ONE: 'NoneType' object is not subscriptable
Error processing finca 001, variety ROYAL DAMASCUS: 'NoneType' object is not subscriptable
Error processing finca 001, variety SORIA: 'NoneType' 

In [85]:
df_all_processed.to_excel(r"C:\Users\ESTADISTICA\Downloads\Curvas para revisar.xlsx")

In [52]:
# Suponiendo que 'df_6' es el DataFrame que deseas exportar
ruta_archivo = "C:/Users/ESTADISTICA/Desktop/py/curvas/files/datos.xlsx"  # Ruta completa del archivo Excel
df_7.to_excel(ruta_archivo, sheet_name='bd_python', index=False)

In [53]:
df_g = df_7.copy()

In [54]:
df_g = df_g.query("nvariedad=='DON PEDRO' and pico==1")

In [55]:
# Función para generar reporte resumido por finca
def generar_reporte_finca(df, nombre_finca):
    reporte = {
        'Finca': nombre_finca,
        'Total Variedades': df['nvariedad'].nunique(),
        'Total Bloques': 0,#df['bloque'].nunique(),
        'Total Cosechas': 0,#df['cosecha'].nunique(),
        'Producción Promedio': df['fm_aj'].mean(),
        'Variedad Mejor Producción': df.groupby('nvariedad')['fm_aj'].mean().idxmax(),
        'Producción Máxima': df['fm_aj'].max(),
        'Semana Pico Más Común': moda_sem_pico[moda_sem_pico['nvariedad'].isin(df['nvariedad'])]['sem_pico'].mode()[0]
    }
    return pd.DataFrame([reporte])

# Ejemplo de generación de reporte
reportes = []
for finca in df_7['finca'].unique():
    reporte = generar_reporte_finca(df_7[df_7['finca'] == finca], finca)
    reportes.append(reporte)

reporte_completo = pd.concat(reportes)
reporte_completo.to_excel("//svrinverpalmas/usuarios/Hilder/Data/reporte_fincas.xlsx", index=False)

In [56]:
# Aplicación de filtro Savitzky-Golay para suavizar curvas
df_curvas = df_2.copy()
ventana = 11  # Tamaño de la ventana para el filtro (debe ser impar)
orden = 3     # Orden del polinomio para el ajuste

# Aplicar filtro por variedad y finca
df_curvas['fm_aj_smooth'] = df_curvas.groupby(['finca', 'nvariedad'])['f_m'].transform(
    lambda x: savgol_filter(x, window_length=ventana, polyorder=orden) if len(x) >= ventana else x
)

In [57]:
df_2.head()

,finca,bloque,nvariedad,cosecha,pico,edad,f_m,sem_pico_x,sem_pico_y,filtro
0,001,23,AMICO LAVANDER,FEB24,1.0,22,0.010326,29.0,28.0,2
1,001,23,AMICO LAVANDER,FEB24,1.0,23,0.107609,29.0,28.0,2
2,001,23,AMICO LAVANDER,FEB24,1.0,24,0.215217,29.0,28.0,2
3,001,23,AMICO LAVANDER,FEB24,1.0,25,0.448913,29.0,28.0,2
4,001,23,AMICO LAVANDER,FEB24,1.0,26,0.533152,29.0,28.0,2
